In [8]:
import sys
sys.path.append('../..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [9]:
date = read_table("select * from sc_gold.dim_date")
state= read_table("select * from sc_gold.dim_state")
sector = read_table("select * from sc_gold.dim_sector")

In [10]:
df = read_table("select * from sc_bronze.datagov_gdp")
df

,state,date,sector,RM(million),growth_yoy
0,Johor,2016-01-01,Agriculture,15029.966,-3.718
1,Johor,2016-01-01,Mining and Quarrying,569.326,19.697
2,Johor,2016-01-01,Manufacturing,34121.545,5.448
3,Johor,2016-01-01,Construction,8978.486,23.515
4,Johor,2016-01-01,Services,56266.132,6.046
...,...,...,...,...,...
715,W.P. Putrajaya,2024-01-01,Agriculture,0.000,0.000
716,W.P. Putrajaya,2024-01-01,Mining and Quarrying,44541.000,1.000
717,W.P. Putrajaya,2024-01-01,Manufacturing,0.000,0.000
718,W.P. Putrajaya,2024-01-01,Construction,0.000,0.000


In [11]:
df = df.merge(
    date[["date", "date_id"]],
    on="date",
    how="left"
)

df = df.merge(
    state[["state", "state_id"]],
    on="state",
    how="left"
)

df = df.merge(
    sector[["sector", "sector_id"]],
    on="sector",
    how="left"
)


df_final = df.drop(columns=["date", "state", "sector"])
id_cols = ["date_id", "state_id", "sector_id"]
df_final = df_final[id_cols + [col for col in df_final.columns if col not in id_cols]]

In [12]:
df_final["gsec_id"] = ["GSEC" + str(i+1).zfill(4) for i in range(len(df_final))]
df_final = df_final[["gsec_id"] + [c for c in df_final.columns if c != "gsec_id"]]
df_final

,gsec_id,date_id,state_id,sector_id,RM(million),growth_yoy
0,GSEC0001,DT001,ST001,SEC001,15029.966,-3.718
1,GSEC0002,DT001,ST001,SEC004,569.326,19.697
2,GSEC0003,DT001,ST001,SEC003,34121.545,5.448
3,GSEC0004,DT001,ST001,SEC002,8978.486,23.515
4,GSEC0005,DT001,ST001,SEC005,56266.132,6.046
...,...,...,...,...,...,...
715,GSEC0716,DT033,ST016,SEC001,0.000,0.000
716,GSEC0717,DT033,ST016,SEC004,44541.000,1.000
717,GSEC0718,DT033,ST016,SEC003,0.000,0.000
718,GSEC0719,DT033,ST016,SEC002,0.000,0.000


In [13]:
write_table(df_final, "sc_gold", "fact_gdp_sector")

Table sc_gold.fact_gdp_sector written successfully.
